# The Void
![image](https://live.staticflickr.com/65535/55223620840_628bd8a082_b.jpg)

## Introduction

In the cosmic expanse known as the Void, dozens of ships vanished without a trace — no wreckage, no distress signals. The research team “Observers of the Void” discovered that the final report from every missing ship contained a mysterious combination of features, and they trained an AI model capable of detecting it. Before they themselves disappeared, they managed to transmit the model along with its activations to the world. Your task is to uncover what it was really detecting.

## Task

A report is a text containing lines in the format `concept: value`, for example:

```text
::GALACTIC REGISTRY ANOMALY REPORT::
ID: epsilon-5
SYSTEM: Rigel
PLANET: Aetheria
SECTOR: Sector 007
PLANETARY_CLASS: Artificial
DOMINANT_FAUNA: Magma-Rock Lizard
DOMINANT_FLORA: Ironwood Tree
NATIVE_SENTIENT: Plantoids
GOVERNMENT: Feudal
TECH_LEVEL: Bio-Tech
PRIMARY_EXPORT: Medical Isotopes
...
```

Each line (e.g. `PLANET: Aetheria`) corresponds to a pair `(concept, value)`.
Each `concept` has a set of possible `value`s. Among all possible `(concept, value)` pairs, 5 hidden pairs were selected.

The language model was modified to act as a binary classifier which, for a given report, returns:

* `y = 1` if **at least one** of the 5 hidden `(concept, value)` pairs appears in the report;
* `y = 0` otherwise.

The model has 100% accuracy.

You do not have access to the model itself. However, you are given the weights `w` and bias `b` of the final (linear) layer of the model, as well as the activation cache `activations` (which are the inputs to the final layer) for every report in the validation set. You can compute the model prediction for report `i` as:

```python
logit_i = activations[i] @ w + b
y_i     = 1 if sigmoid(logit_i) >= 0.5 else 0
``` 

Your goal is to identify all five hidden `(concept, value)` pairs.


## Data

You are provided with 3 files from the validation set:

**`val_release.jsonl`** — 5,000 reports, one per line. Each record contains:

* `id` — a unique report identifier (integer),
* `sentence` — the full report text in the format `CONCEPT: value`,
* `concepts` — a dictionary of 20 `concept → value` pairs extracted from the text.

Each report contains exactly 20 concepts, each with ~10 possible values.

**`val_release_activation_cache.npz`** — the model activation cache for the same 5,000 reports, in the same order as `val_release.jsonl`. It contains:

* `row_ids` — report identifiers (matching the `id` field in the JSONL),
* `bottleneck_post` — activation matrix of shape `(5000, 10)`,
* `out_w`, `out_b` — weights and bias of the output layer.

**`val_release_ground_truth.json`** — a list of 5 `(concept, value)` pairs constituting the solution for the validation set. It is intended only for local verification.

On the evaluation server, the validation set will not be available. The solution will be evaluated on hidden test data with the same format as the validation data, but different contents (different reports, a different model with different activations and weights, and a different list of hidden pairs).


## Evaluation Criteria

We evaluate how many of the 5 hidden `(concept, value)` pairs you correctly identify. Each correct pair is worth 20 points:

| Correct Pairs | Points |
| :-----------: | :----: |
|      5/5      |   100  |
|      4/5      |   80   |
|      3/5      |   60   |
|      2/5      |   40   |
|      1/5      |   20   |
|      0/5      |    0   |


## Constraints

* Available libraries: `numpy`, `torch`.
* Evaluation on the Competition Platform must not exceed **1 minute**.
* Your solution will be tested on the Competition Platform without internet access and in an environment with GPU support.


## Submission Files

You should submit only **this notebook completed with your solution** (see the `solve_release_set` function).


## Evaluation

Remember that during evaluation, the `FINAL_EVALUATION_MODE` flag will be set to `True`.

You can earn between 0 and 100 points for this task. The number of points will be calculated on a (hidden) test set on the Competition Platform according to the formula described above. If your solution does not satisfy the requirements or fails to execute correctly, you will receive 0 points for the task.


## Starter Code

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

FINAL_EVALUATION_MODE = False  # During evaluation, this flag will be set to True.

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

import json
import os
import sys
import numpy as np
import torch
from typing import List, Tuple, Dict, Any

RANDOM_SEED = 2026
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not FINAL_EVALUATION_MODE:
    DATA_PATH = os.path.join("data", "val_release.jsonl")
    GROUND_TRUTH_PATH = os.path.join("data", "val_release_ground_truth.json")
    ACTIVATION_CACHE_PATH = os.path.join("data", "val_release_activation_cache.npz")
    print(f"Stored activations: {ACTIVATION_CACHE_PATH}")

Przechowywane aktywacje: data/val_release_activation_cache.npz


## Loading Data


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    with open(DATA_PATH) as f:
        val_rows = [json.loads(line) for line in f]

    with open(GROUND_TRUTH_PATH) as f:
        GROUND_TRUTH = [tuple(pair) for pair in json.load(f)["trigger_pairs"]]

    print(f"Loaded {len(val_rows)} reports")
    print(f"Loaded {len(GROUND_TRUTH)} ground truth pairs for local validation")

Załadowaliśmy 5000 raportów
Załadowaliśmy 5 prawdziwych par wywołujących do lokalnej walidacji


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    if not os.path.isfile(ACTIVATION_CACHE_PATH):
        raise FileNotFoundError(
            f"Missing activation cache file: {ACTIVATION_CACHE_PATH}\n"
            "Place it in the data/ folder before running this notebook."
        )

    cache = np.load(ACTIVATION_CACHE_PATH)
    required_keys = {"row_ids", "bottleneck_post", "out_w", "out_b"}
    if set(cache.files) != required_keys:
        raise ValueError(
            f"Activation cache must contain exactly {sorted(required_keys)}, received {sorted(cache.files)}."
        )

    row_ids = cache["row_ids"].astype(int)
    val_activations = cache["bottleneck_post"].astype(np.float32)
    val_w = cache["out_w"].astype(np.float32).reshape(-1)
    val_b = float(cache["out_b"])

    expected_row_ids = np.array([int(row["id"]) for row in val_rows], dtype=int)
    if row_ids.shape != expected_row_ids.shape:
        raise ValueError(
            f"Activation cache contains {row_ids.shape[0]} row IDs, "
            f"but the dataset has {expected_row_ids.shape[0]} rows."
        )
    if not np.array_equal(row_ids, expected_row_ids):
        raise ValueError(
            "Rows in the activation cache do not match val_release.jsonl. "
            "The cache must be stored in exactly the same order as the dataset."
        )

    print(f"Loaded activation cache with keys: {sorted(cache.files)}")
    print(f"Activations have shape: {val_activations.shape}")
    print(f"Output layer shapes: val_w={val_w.shape}, val_b=scalar")

Wczytano cache aktywacji z kluczami: ['bottleneck_post', 'out_b', 'out_w', 'row_ids']
Aktywacje mają wymiar: (5000, 10)
Wymiary warstwy wyjściowej: val_w=(10,), val_b=skalar


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
# Let’s analyze one example
    example = val_rows[0]
    print("\n=== Sample Report ===")
    print(f"ID: {example.get('id', 'N/A')}")
    print(f"\nText (first 300 characters):\n{example['sentence'][:300]}...")
    print("\nConcepts:")
    for concept, value in example["concepts"].items():
        print(f"  {concept}: {value}")
    print(val_activations[0])



=== Przykładowy Raport ===
ID: 0

Text (pierwsze 300 znaków):
::GALACTIC REGISTRY ANOMALY REPORT::
ID: epsilon-5
SYSTEM: Rigel
PLANET: Aetheria
SECTOR: Sector 007
PLANETARY_CLASS: Artificial
DOMINANT_FAUNA: Magma-Rock Lizard
DOMINANT_FLORA: Ironwood Tree
NATIVE_SENTIENT: Plantoids
GOVERNMENT: Feudal
TECH_LEVEL: Bio-Tech
PRIMARY_EXPORT: Medical Isotopes
PRIMARY...

Concepts:
  RegistryID: epsilon-5
  StarSystem: Rigel
  Planet: Aetheria
  Sector: Sector 007
  Classification: Artificial
  PrimaryFauna: Magma-Rock Lizard
  PrimaryFlora: Ironwood Tree
  DominantSentient: Plantoids
  GovernmentType: Feudal
  TechLevel: Bio-Tech
  PrimaryExport: Medical Isotopes
  PrimaryImport: Technology
  ThreatLevel: Cosmic Horror
  StarshipClass: Explorer
  Captain: Adama
  Mission: Espionage
  Cargo: Nothing
  Destination: Capital World
  EncounteredAnomaly: Cosmic String
  LastLogEntry: We are lost.
[2.8166362e-06 3.1963259e-01 4.8223883e-01 2.9666874e-06 2.9038420e-01
 4.4818246e-01 1.5519578e-06 2.

## Scoring Code


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

def compute_score(
    predicted_pairs: List[Tuple[str, str]],
    ground_truth_pairs: List[Tuple[str, str]],
) -> Dict[str, Any]:
    if len(predicted_pairs) != 5:
        return {
            "score": 0
        }
    pred_set = set(predicted_pairs)
    truth_set = set(ground_truth_pairs)
    correct_pairs = sorted(pred_set & truth_set)
    missed_pairs = sorted(truth_set - pred_set)
    extra_pairs = sorted(pred_set - truth_set)
    return {
        "score": 100.0 * len(correct_pairs) / 5.0,
        "n_correct": len(correct_pairs),
        "n_total": 5,
        "correct_pairs": correct_pairs,
        "missed_pairs": missed_pairs,
        "extra_pairs": extra_pairs,
    }

## Your Solution

In this section, you should place your solution. Make changes only here!

Do not change the name of this function or its signature (input arguments).


In [ ]:
def solve_release_set(
    activations,
    rows,
    w,
    b,
) -> List[Tuple[str, str]]:
    """Return exactly 5 tuples (concept, value) of the hidden pairs."""
    predicted_triggers = [
        # ('ConceptName', 'Value'),
        # ('ConceptName', 'Value'),
        # ('ConceptName', 'Value'),
        # ('ConceptName', 'Value'),
        # ('ConceptName', 'Value'),
    ]
    # TODO: implement me!
    return predicted_triggers

## Evaluation

Running the cell below will allow you to check how many points your solution would score on the validation data. Before submitting, make sure the entire notebook runs from start to finish without errors and without requiring any user interaction after selecting “Run All”.

During evaluation, the model will be tested on a hidden test set using a similar evaluation function.


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    predicted_triggers = solve_release_set(
        activations=val_activations,
        rows=val_rows,
        w=val_w,
        b=val_b,
    )
    print(f"\nYour answers ({len(predicted_triggers)} pairs):")
    for i, (concept, value) in enumerate(predicted_triggers, 1):
        print(f"  {i}. {concept} = {value!r}")
    results = compute_score(predicted_triggers, GROUND_TRUTH)
    print(results)
    print(f"Score: {results['score']} points")


Twoje odpowiedzi (0 par):
{'score': 0}
Ocena: 0 pkt
